<a href="https://colab.research.google.com/github/kimheeseo/LSCNS/blob/main/gn_hcf_two_paper_independent_validation_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GN model + HCF geometry model: paper-input independent validation

## 목적

논문에 공개된 **입력조건만** 코드에 넣고, 논문의 비교 출력값은 보정에 사용하지 않은 상태로 코드 성능을 평가한다.

- GN: Poggiolini (2012)의 Table I 및 기준 시스템 조건으로 repository closed form을 평가한다.
- HCF: Petrovich et al. (2025)의 HCF2 기하 범위로 Raw Hasan–modified MS 및 Raw bouncing-ray 모델을 평가한다.
- 논문 출력값은 계산이 끝난 뒤에만 오차 계산에 사용한다.
- `f1/f2` least-squares 보정과 hybrid-loss target fitting은 실행하지 않는다.
- 절대오차율 10%는 사용자가 확인하기 위한 표시 기준이며 논문이 정한 합격 기준은 아니다.

## 중요한 검증 수준

- GN Eq.13 비교: 이론식 구현 일관성 검증이며 독립 전송실험 검증은 아니다.
- GN 최적전력 비교: 논문에 명시된 세 수치 예제와의 비교다.
- HCF 분산 비교: 현재 해석모델과 논문의 FEM 모델 결과 비교다.
- HCF 손실 비교: 현재 Raw bouncing-ray와 논문의 측정 총손실 비교다. 두 값의 물리적 범위가 다르므로 모델 적용성 진단으로 해석한다.

## 1차 출처

1. P. Poggiolini, JLT 30(24), 3857–3879 (2012), DOI: 10.1109/JLT.2012.2217729.
2. M. Petrovich et al., Nature Photonics 19, 1203–1208 (2025), DOI: 10.1038/s41566-025-01747-5; arXiv:2503.21467.


In [ ]:
import warnings
from dataclasses import dataclass
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

C0 = 299_792_458.0
H_PLANCK = 6.626_070_15e-34
U01 = 2.4048255577
DB_PER_NEPER = 10.0 / np.log(10.0)
ERROR_FLAG_PERCENT = 10.0

def signed_error_percent(model, reference):
    model = np.asarray(model, dtype=float)
    reference = np.asarray(reference, dtype=float)
    return 100.0 * (model - reference) / reference

def mape_percent(model, reference):
    return float(np.mean(np.abs(signed_error_percent(model, reference))))

def rmse(model, reference):
    model = np.asarray(model, dtype=float)
    reference = np.asarray(reference, dtype=float)
    return float(np.sqrt(np.mean((model-reference)**2)))

print("Independent validation notebook initialized.")
print("Paper-output calibration: OFF")


## 1. GN 모델 검증

Poggiolini 논문의 입력조건:

- Table I: LPSCF, SMF, NZDSF의 감쇠·분산·비선형계수
- 파장 1550 nm
- Nyquist-WDM: 32 GBd, 32 GHz, 157채널
- 기준 span: 100 km
- 최적전력 예제: RS-SMF 100 km, NY-SMF 100 km, NY-SMF 75 km
- EDFA noise figure: 6 dB

Eq.13 값은 논문 식을 입력조건으로 다시 계산한 benchmark이다. 논문에 표로 제시된 독립 측정 `eta` 값은 아니다.


In [ ]:
@dataclass(frozen=True)
class GNFiber:
    name: str
    attenuation_db_km: float
    dispersion_ps_nm_km: float
    gamma_per_w_km: float

GN_FIBERS = [
    GNFiber("LPSCF", 0.165, 20.4, 0.8),
    GNFiber("SMF",   0.200, 16.5, 1.3),
    GNFiber("NZDSF", 0.200,  3.9, 1.6),
]

GN_CONDITIONS = {
    "wavelength_nm": 1550.0,
    "span_length_km": 100.0,
    "symbol_rate_gbd": 32.0,
    "spacing_ghz": 32.0,
    "n_channels": 157,
    "n_spans": 1,
    "spectrum": "ideal rectangular Nyquist-WDM",
}

display(pd.DataFrame([vars(f) for f in GN_FIBERS]))
display(pd.DataFrame(
    list(GN_CONDITIONS.items()), columns=["Paper input", "Value"]
))


In [ ]:
def beta2_s2_per_km(dispersion_ps_nm_km, wavelength_nm=1550.0):
    wavelength_m = wavelength_nm*1e-9
    dispersion_si = np.asarray(dispersion_ps_nm_km, dtype=float)*1e-6
    return np.abs(
        -(wavelength_m**2/(2.0*np.pi*C0))*dispersion_si*1000.0
    )

def repository_closed_form_eta(
    fiber, symbol_rate_hz, spacing_hz, n_channels, n_spans=1
):
    beta2 = beta2_s2_per_km(fiber.dispersion_ps_nm_km)
    alpha_field = fiber.attenuation_db_km/8.685889638
    argument = (
        np.pi**2*beta2*symbol_rate_hz**2/(4.0*alpha_field)
        * n_channels**(2.0*symbol_rate_hz/spacing_hz)
    )
    return (
        n_spans*4.0*fiber.gamma_per_w_km**2
        /(27.0*np.pi*beta2*alpha_field*symbol_rate_hz**2)
        * np.arcsinh(argument)
    )

def poggiolini_eq13_eta(
    fiber, span_length_km, symbol_rate_hz, n_channels
):
    beta2 = beta2_s2_per_km(fiber.dispersion_ps_nm_km)
    alpha_field = fiber.attenuation_db_km/8.685889638
    leff = (
        1.0-np.exp(-2.0*alpha_field*span_length_km)
    )/(2.0*alpha_field)
    leff_asymptotic = 1.0/(2.0*alpha_field)
    bandwidth_hz = n_channels*symbol_rate_hz
    gnli_over_gwdm_cubed = (
        8.0/27.0*fiber.gamma_per_w_km**2*leff**2
        * np.arcsinh(
            0.5*np.pi**2*beta2*leff_asymptotic*bandwidth_hz**2
        )
        /(np.pi*beta2*leff_asymptotic)
    )
    return gnli_over_gwdm_cubed/symbol_rate_hz**2

rs = GN_CONDITIONS["symbol_rate_gbd"]*1e9
spacing = GN_CONDITIONS["spacing_ghz"]*1e9

gn_eta_rows = []
for fiber in GN_FIBERS:
    paper_equation = poggiolini_eq13_eta(
        fiber, GN_CONDITIONS["span_length_km"], rs,
        GN_CONDITIONS["n_channels"]
    )
    current_code = repository_closed_form_eta(
        fiber, rs, spacing, GN_CONDITIONS["n_channels"],
        GN_CONDITIONS["n_spans"]
    )
    error = float(signed_error_percent(current_code, paper_equation))
    gn_eta_rows.append({
        "Fiber": fiber.name,
        "Poggiolini Eq.13 eta (1/W²)": paper_equation,
        "Repository eta (1/W²)": current_code,
        "Signed error (%)": error,
        "Absolute error (%)": abs(error),
        "10% flag": abs(error) >= ERROR_FLAG_PERCENT,
        "Validation type": "equation implementation consistency",
    })

gn_eta_comparison = pd.DataFrame(gn_eta_rows)
gn_eta_mape = gn_eta_comparison["Absolute error (%)"].mean()
display(gn_eta_comparison)
print(f"GN eta MAPE = {gn_eta_mape:.3f}%")


In [ ]:
@dataclass(frozen=True)
class LaunchCase:
    name: str
    n_channels: int
    spacing_ghz: float
    span_km: float
    paper_popt_dbm: float

PAPER_LAUNCH_CASES = [
    LaunchCase("RS-SMF, 100 km", 101, 50.0, 100.0, -0.4),
    LaunchCase("NY-SMF, 100 km", 157, 32.0, 100.0, -1.0),
    LaunchCase("NY-SMF, 75 km",  157, 32.0,  75.0, -2.6),
]

def repository_optimum_launch_dbm(case, noise_figure_db=6.0):
    fiber = next(f for f in GN_FIBERS if f.name == "SMF")
    eta = repository_closed_form_eta(
        fiber, 32e9, case.spacing_ghz*1e9, case.n_channels, 1
    )
    frequency_hz = C0/1550e-9
    noise_factor = 10.0**(noise_figure_db/10.0)
    gain = 10.0**(
        fiber.attenuation_db_km*case.span_km/10.0
    )
    ase_w = (
        H_PLANCK*frequency_hz*noise_factor*32e9*(gain-1.0)
    )
    popt_w = (ase_w/(2.0*eta))**(1.0/3.0)
    return 10.0*np.log10(popt_w)+30.0, popt_w, eta, ase_w

popt_rows = []
for case in PAPER_LAUNCH_CASES:
    model_dbm, model_w, eta, ase_w = repository_optimum_launch_dbm(case)
    paper_w = 10.0**((case.paper_popt_dbm-30.0)/10.0)
    linear_error = float(signed_error_percent(model_w, paper_w))
    popt_rows.append({
        "Case": case.name,
        "Paper Popt (dBm/ch)": case.paper_popt_dbm,
        "Code Popt (dBm/ch)": model_dbm,
        "Difference (dB)": model_dbm-case.paper_popt_dbm,
        "Linear-power error (%)": linear_error,
        "Absolute power error (%)": abs(linear_error),
        "10% flag": abs(linear_error) >= ERROR_FLAG_PERCENT,
    })

gn_popt_comparison = pd.DataFrame(popt_rows)
gn_popt_mape = gn_popt_comparison["Absolute power error (%)"].mean()
display(gn_popt_comparison)
print(f"GN optimum-power linear MAPE = {gn_popt_mape:.3f}%")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

x = np.arange(len(gn_eta_comparison))
w = 0.36
axes[0].bar(
    x-w/2, gn_eta_comparison["Poggiolini Eq.13 eta (1/W²)"],
    width=w, label="Poggiolini Eq.13"
)
axes[0].bar(
    x+w/2, gn_eta_comparison["Repository eta (1/W²)"],
    width=w, label="Repository code"
)
axes[0].set_xticks(x, gn_eta_comparison["Fiber"])
axes[0].set_yscale("log")
axes[0].set_ylabel("eta (1/W²)")
axes[0].set_title(f"GN eta comparison — MAPE {gn_eta_mape:.2f}%")
axes[0].legend()

x = np.arange(len(gn_popt_comparison))
paper_p = gn_popt_comparison["Paper Popt (dBm/ch)"].to_numpy()
model_p = gn_popt_comparison["Code Popt (dBm/ch)"].to_numpy()
axes[1].bar(x-w/2, paper_p, width=w, label="Paper")
axes[1].bar(x+w/2, model_p, width=w, label="Repository code")
axes[1].set_xticks(
    x, ["RS 100", "NY 100", "NY 75"]
)
axes[1].set_ylabel("Popt (dBm/channel)")
axes[1].set_title(
    f"Optimum launch comparison — linear MAPE {gn_popt_mape:.2f}%"
)
axes[1].legend()
for i, err in enumerate(gn_popt_comparison["Linear-power error (%)"]):
    axes[1].text(
        i, max(paper_p[i], model_p[i])+0.1, f"{err:+.1f}%",
        ha="center", fontsize=9
    )

fig.savefig("gn_model_paper_vs_code.png", bbox_inches="tight")
plt.show()


## 2. HCF geometry model 독립 비교

Petrovich HCF2 공개 입력:

- 코어 직경: 29.1–29.6 µm; nominal 모델값 29.5 µm 사용
- 큰 튜브: 30.4–31.7 µm; 중앙값 31.05 µm 사용
- 중간 튜브: 22.7–24.8 µm
- 작은 튜브: 7.0–8.4 µm
- 각 막 두께: 약 0.50 µm
- 외부 튜브 세트: 5
- 이중 중첩 구조

현재 Hasan/MS 코드는 중간·작은 튜브 직경을 직접 사용하지 않는다. 따라서 이는 정확한 SEM-contour FEM 재현이 아니라 단순화된 기하모델의 독립 성능 평가다.

논문 비교 출력:

- FEM 모델 분산: 2.1, 3.2, 3.7 ps/(nm·km) at 1310, 1550, 1700 nm
- 측정 평균손실: 0.128, 0.091 dB/km at 1310, 1550 nm
- 이상적 동일 nominal 구조의 모델손실: 0.07 dB/km at 1550 nm


In [ ]:
@dataclass(frozen=True)
class DNANFGeometry:
    core_radius_um: float = 14.75
    membrane_thickness_um: float = 0.50
    outer_tube_diameter_um: float = 31.05
    middle_tube_diameter_um: float = 23.75
    inner_tube_diameter_um: float = 7.70
    tube_count: int = 5
    nesting_order: int = 2

    @property
    def core_radius_m(self):
        return self.core_radius_um*1e-6

    @property
    def membrane_thickness_m(self):
        return self.membrane_thickness_um*1e-6

    @property
    def perimeter_gap_m(self):
        theta = np.pi/self.tube_count
        return (
            2.0*self.core_radius_m*np.sin(theta)
            - self.outer_tube_diameter_um*1e-6*(1.0-np.sin(theta))
        )

geometry = DNANFGeometry()

hcf_input_table = pd.DataFrame([
    ["Core diameter","29.1–29.6 µm",2*geometry.core_radius_um,"Used"],
    ["Large tube diameter","30.4–31.7 µm",geometry.outer_tube_diameter_um,"Used"],
    ["Middle tube diameter","22.7–24.8 µm",geometry.middle_tube_diameter_um,"Recorded; not used by current equation"],
    ["Small tube diameter","7.0–8.4 µm",geometry.inner_tube_diameter_um,"Recorded; not used by current equation"],
    ["Membrane thickness","~0.50 µm",geometry.membrane_thickness_um,"Used"],
    ["Outer tube count","5",geometry.tube_count,"Used"],
    ["Nesting order","double nested",geometry.nesting_order,"Used in Hasan coefficient"],
], columns=["Input","Paper HCF2","Code value","Current-model use"])
display(hcf_input_table)


In [ ]:
def hasan_radius_coefficients(geometry):
    radius_to_gap = geometry.core_radius_m/geometry.perimeter_gap_m
    n = geometry.tube_count
    nesting = geometry.nesting_order
    a0, a1 = 0.097041, 1.095
    b0, b1, b2, b3 = 0.76246, 0.007584, 0.002, 0.012
    f1 = a1*np.exp(a0/radius_to_gap)
    f2 = (
        b1*n*np.exp(b0/radius_to_gap)-b2*n+b3
        +0.0045*np.exp(-4.1589/(nesting*radius_to_gap))
    )
    return float(f1), float(f2)

def effective_radius_m(wavelength_m, geometry, f1, f2):
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    return f1*geometry.core_radius_m*(
        1.0-f2*wavelength_m**2/
        (geometry.core_radius_m*geometry.membrane_thickness_m)
    )

def effective_index(wavelength_m, geometry, f1, f2):
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    reff = effective_radius_m(wavelength_m, geometry, f1, f2)
    return 1.0-0.125*(
        U01*wavelength_m/(np.pi*reff)
    )**2

def chromatic_dispersion_ps_nm_km(
    wavelength_m, geometry, derivative_step_nm=0.20
):
    f1, f2 = hasan_radius_coefficients(geometry)
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    h = derivative_step_nm*1e-9
    def n_eff(offset):
        return effective_index(
            wavelength_m+offset, geometry, f1, f2
        )
    d2n = (
        -n_eff(2*h)+16*n_eff(h)-30*n_eff(0.0)
        +16*n_eff(-h)-n_eff(-2*h)
    )/(12*h**2)
    return -(wavelength_m/C0)*d2n/1e-6

# Guard against accidental paper-output calibration.
assert "least_squares" not in chromatic_dispersion_ps_nm_km.__code__.co_names

PAPER_D_WAVELENGTH_NM = np.array([1310.0, 1550.0, 1700.0])
PAPER_D = np.array([2.1, 3.2, 3.7])
RAW_D = chromatic_dispersion_ps_nm_km(
    PAPER_D_WAVELENGTH_NM*1e-9, geometry
)

# Sensitivity envelope uses only the geometry ranges reported in the paper.
geometry_samples = []
sample_predictions = []
for core_diameter_um in [29.1, 29.35, 29.6]:
    for outer_diameter_um in [30.4, 31.05, 31.7]:
        sample = DNANFGeometry(
            core_radius_um=core_diameter_um/2.0,
            membrane_thickness_um=0.50,
            outer_tube_diameter_um=outer_diameter_um,
            middle_tube_diameter_um=23.75,
            inner_tube_diameter_um=7.70,
            tube_count=5,
            nesting_order=2,
        )
        if sample.perimeter_gap_m > 0:
            geometry_samples.append(sample)
            sample_predictions.append(
                chromatic_dispersion_ps_nm_km(
                    PAPER_D_WAVELENGTH_NM*1e-9, sample
                )
            )
sample_predictions = np.asarray(sample_predictions)
d_lower = sample_predictions.min(axis=0)
d_upper = sample_predictions.max(axis=0)

d_error = signed_error_percent(RAW_D, PAPER_D)
hcf_dispersion_comparison = pd.DataFrame({
    "Wavelength (nm)": PAPER_D_WAVELENGTH_NM,
    "Paper FEM D (ps/nm/km)": PAPER_D,
    "Raw code D (ps/nm/km)": RAW_D,
    "Geometry-range minimum": d_lower,
    "Geometry-range maximum": d_upper,
    "Signed error (%)": d_error,
    "Absolute error (%)": np.abs(d_error),
    "10% flag": np.abs(d_error) >= ERROR_FLAG_PERCENT,
    "Paper value inside geometry envelope": (
        (PAPER_D >= d_lower) & (PAPER_D <= d_upper)
    ),
})
hcf_d_mape = mape_percent(RAW_D, PAPER_D)
hcf_d_rmse = rmse(RAW_D, PAPER_D)
display(hcf_dispersion_comparison)
print(f"HCF raw-dispersion MAPE = {hcf_d_mape:.3f}%")
print(f"HCF raw-dispersion RMSE = {hcf_d_rmse:.4f} ps/(nm km)")


In [ ]:
# Finite-difference convergence check at 1310 nm.
step_grid_nm = np.array([0.05, 0.10, 0.20, 0.50, 1.00])
step_values = np.array([
    chromatic_dispersion_ps_nm_km(
        1310e-9, geometry, derivative_step_nm=step
    )
    for step in step_grid_nm
])
derivative_audit = pd.DataFrame({
    "Finite-difference step (nm)": step_grid_nm,
    "D at 1310 nm (ps/nm/km)": step_values,
})
step_spread = float(step_values.max()-step_values.min())
display(derivative_audit)
print(f"Step sensitivity spread = {step_spread:.6f} ps/(nm km)")
assert step_spread < 0.01


In [ ]:
def silica_index_sellmeier(wavelength_m):
    wavelength_um = np.asarray(wavelength_m, dtype=float)*1e6
    wavelength_sq = wavelength_um**2
    b = np.array([0.6961663, 0.4079426, 0.8974794])
    c_um = np.array([0.0684043, 0.1162414, 9.896161])
    n_sq = np.ones_like(wavelength_sq)
    for bi, ci in zip(b, c_um):
        n_sq += bi*wavelength_sq/(wavelength_sq-ci**2)
    return np.sqrt(n_sq)

def capillary_bouncing_ray_loss_db_km(wavelength_m, geometry):
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    k0 = 2*np.pi/wavelength_m
    radius = geometry.core_radius_m
    wall = geometry.membrane_thickness_m
    kappa = U01/radius
    silica_n = silica_index_sellmeier(wavelength_m)
    sigma = k0*np.sqrt(silica_n**2-1.0)
    phase = sigma*wall
    te_den = (
        4*np.cos(phase)**2
        +(kappa/sigma+sigma/kappa)**2*np.sin(phase)**2
    )
    tm_den = (
        4*np.cos(phase)**2
        +(silica_n**2*kappa/sigma
          +sigma/(silica_n**2*kappa))**2*np.sin(phase)**2
    )
    alpha_te = 2*U01/(radius**2*k0*te_den)
    alpha_tm = 2*U01/(radius**2*k0*tm_den)
    return 0.5*(alpha_te+alpha_tm)*DB_PER_NEPER*1000.0

PAPER_LOSS_WAVELENGTH_NM = np.array([1310.0, 1550.0])
PAPER_MEASURED_LOSS = np.array([0.128, 0.091])
RAW_BR_LOSS = capillary_bouncing_ray_loss_db_km(
    PAPER_LOSS_WAVELENGTH_NM*1e-9, geometry
)
loss_error = signed_error_percent(RAW_BR_LOSS, PAPER_MEASURED_LOSS)

hcf_loss_comparison = pd.DataFrame({
    "Wavelength (nm)": PAPER_LOSS_WAVELENGTH_NM,
    "Paper measured total loss (dB/km)": PAPER_MEASURED_LOSS,
    "Raw bouncing-ray code (dB/km)": RAW_BR_LOSS,
    "Signed error (%)": loss_error,
    "Absolute error (%)": np.abs(loss_error),
    "10% flag": np.abs(loss_error) >= ERROR_FLAG_PERCENT,
    "Interpretation": [
        "Model-layer mismatch: raw leakage proxy vs measured total loss",
        "Model-layer mismatch: raw leakage proxy vs measured total loss",
    ],
})
hcf_loss_mape = mape_percent(RAW_BR_LOSS, PAPER_MEASURED_LOSS)
hcf_loss_rmse = rmse(RAW_BR_LOSS, PAPER_MEASURED_LOSS)

ideal_loss_error = float(
    signed_error_percent(RAW_BR_LOSS[1], 0.070)
)
display(hcf_loss_comparison)
print(f"Raw bouncing-ray vs measured-loss MAPE = {hcf_loss_mape:.3e}%")
print(f"Raw bouncing-ray loss RMSE = {hcf_loss_rmse:.3e} dB/km")
print(
    "1550-nm raw bouncing-ray vs paper ideal-geometry 0.070 dB/km "
    f"error = {ideal_loss_error:.3e}%"
)
print("Assessment: current raw bouncing-ray is not a usable DNANF total-loss predictor.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

x = np.arange(len(PAPER_D_WAVELENGTH_NM))
w = 0.36
axes[0].bar(x-w/2, PAPER_D, width=w, label="Paper FEM")
axes[0].bar(x+w/2, RAW_D, width=w, label="Raw Hasan/MS")
axes[0].errorbar(
    x+w/2, RAW_D,
    yerr=np.vstack([RAW_D-d_lower, d_upper-RAW_D]),
    fmt="none", ecolor="black", capsize=4,
    label="Paper geometry-range sensitivity"
)
axes[0].set_xticks(x, PAPER_D_WAVELENGTH_NM.astype(int))
axes[0].set_xlabel("Wavelength (nm)")
axes[0].set_ylabel("D (ps/(nm km))")
axes[0].set_title(
    f"HCF dispersion — MAPE {hcf_d_mape:.2f}%"
)
axes[0].legend()
for i, err in enumerate(d_error):
    axes[0].text(
        i, max(PAPER_D[i], RAW_D[i])+0.10,
        f"{err:+.1f}%", ha="center", fontsize=9
    )

x = np.arange(len(PAPER_LOSS_WAVELENGTH_NM))
axes[1].bar(
    x-w/2, PAPER_MEASURED_LOSS, width=w,
    label="Paper measured total loss"
)
axes[1].bar(
    x+w/2, RAW_BR_LOSS, width=w,
    label="Raw bouncing-ray code"
)
axes[1].set_xticks(x, PAPER_LOSS_WAVELENGTH_NM.astype(int))
axes[1].set_yscale("log")
axes[1].set_xlabel("Wavelength (nm)")
axes[1].set_ylabel("Loss (dB/km, log scale)")
axes[1].set_title("HCF loss — diagnostic model mismatch")
axes[1].legend(fontsize=8)

fig.savefig("hcf_geometry_paper_vs_code.png", bbox_inches="tight")
plt.show()


In [ ]:
# Error-only plots. Loss uses a logarithmic axis because its mismatch is extreme.
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), constrained_layout=True)

axes[0].bar(
    gn_eta_comparison["Fiber"],
    gn_eta_comparison["Absolute error (%)"],
    color="#4c78a8"
)
axes[0].axhline(10, color="red", ls="--", lw=1)
axes[0].set_title("GN eta error")
axes[0].set_ylabel("Absolute error (%)")

axes[1].bar(
    ["RS100","NY100","NY75"],
    gn_popt_comparison["Absolute power error (%)"],
    color="#59a14f"
)
axes[1].axhline(10, color="red", ls="--", lw=1)
axes[1].set_title("GN Popt error")

axes[2].bar(
    PAPER_D_WAVELENGTH_NM.astype(int).astype(str),
    np.abs(d_error), color="#f28e2b"
)
axes[2].axhline(10, color="red", ls="--", lw=1)
axes[2].set_title("HCF D error")
axes[2].set_xlabel("Wavelength (nm)")

axes[3].bar(
    PAPER_LOSS_WAVELENGTH_NM.astype(int).astype(str),
    np.abs(loss_error), color="#e15759"
)
axes[3].set_yscale("log")
axes[3].axhline(10, color="red", ls="--", lw=1)
axes[3].set_title("HCF loss error")
axes[3].set_xlabel("Wavelength (nm)")

fig.savefig("paper_code_absolute_error.png", bbox_inches="tight")
plt.show()


## 3. 종합 판정

이 노트북의 독립성은 다음 조건으로 강제된다.

1. HCF 논문 분산값은 `PAPER_D` 비교 배열에만 존재하며 (f_1,f_2) 계산에는 들어가지 않는다.
2. HCF 측정손실은 오차 계산에만 사용하며 손실계수 피팅은 없다.
3. GN의 논문 최적전력값은 비교 테이블에만 사용하며 GN/ASE 계산 입력에는 들어가지 않는다.
4. 그래프와 MAPE/RMSE는 모델 계산이 끝난 뒤 생성한다.


In [ ]:
summary = pd.DataFrame([
    {
        "Model": "GN repository eta",
        "Reference": "Poggiolini Eq.13 recomputed from paper inputs",
        "Metric": "MAPE",
        "Value": gn_eta_mape,
        "Unit": "%",
        "Validation level": "Equation implementation consistency",
    },
    {
        "Model": "GN optimum launch",
        "Reference": "Three numerical values stated in paper",
        "Metric": "Linear-power MAPE",
        "Value": gn_popt_mape,
        "Unit": "%",
        "Validation level": "Published numerical-example comparison",
    },
    {
        "Model": "Raw Hasan/MS dispersion",
        "Reference": "Petrovich HCF2 FEM dispersion",
        "Metric": "MAPE",
        "Value": hcf_d_mape,
        "Unit": "%",
        "Validation level": "No-fit model-to-model comparison",
    },
    {
        "Model": "Raw Hasan/MS dispersion",
        "Reference": "Petrovich HCF2 FEM dispersion",
        "Metric": "RMSE",
        "Value": hcf_d_rmse,
        "Unit": "ps/(nm km)",
        "Validation level": "No-fit model-to-model comparison",
    },
    {
        "Model": "Raw bouncing-ray loss",
        "Reference": "Petrovich HCF2 measured total loss",
        "Metric": "MAPE",
        "Value": hcf_loss_mape,
        "Unit": "%",
        "Validation level": "Independent diagnostic; model-layer mismatch",
    },
])

display(summary)

print("=== EVIDENCE-BASED CONCLUSION ===")
print(f"GN eta equation-consistency MAPE: {gn_eta_mape:.3f}%")
print(f"GN optimum-launch linear MAPE: {gn_popt_mape:.3f}%")
print(f"HCF raw-dispersion MAPE: {hcf_d_mape:.3f}%")
print(f"HCF raw-dispersion RMSE: {hcf_d_rmse:.4f} ps/(nm km)")
print(f"HCF raw-loss diagnostic MAPE: {hcf_loss_mape:.3e}%")
print("- GN results support preliminary implementation consistency, not independent experiment accuracy.")
print("- Raw HCF dispersion is reasonable near 1550/1700 nm but misses 1310 nm by more than 10%.")
print("- Raw bouncing-ray cannot represent HCF2 total loss; LL+SSL+microbend/FEM extension is required.")
print("- No capacity validation is claimed because neither selected paper supplies matching capacity outputs for this exact code setup.")

# Reproducibility and leakage guards.
assert np.isfinite(summary["Value"]).all()
assert not np.any(np.isclose(RAW_D[:, None], PAPER_D[None, :], rtol=0, atol=0))
assert "least_squares" not in globals()
print("CALIBRATION LEAKAGE AUDIT: PASS")


In [ ]:
gn_eta_comparison.to_csv("gn_eta_comparison.csv", index=False)
gn_popt_comparison.to_csv("gn_optimum_launch_comparison.csv", index=False)
hcf_dispersion_comparison.to_csv(
    "hcf_raw_dispersion_comparison.csv", index=False
)
hcf_loss_comparison.to_csv(
    "hcf_raw_loss_comparison.csv", index=False
)
derivative_audit.to_csv("hcf_derivative_audit.csv", index=False)
summary.to_csv("two_paper_independent_validation_summary.csv", index=False)

DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    archive_files = [
        "gn_model_paper_vs_code.png",
        "hcf_geometry_paper_vs_code.png",
        "paper_code_absolute_error.png",
        "gn_eta_comparison.csv",
        "gn_optimum_launch_comparison.csv",
        "hcf_raw_dispersion_comparison.csv",
        "hcf_raw_loss_comparison.csv",
        "hcf_derivative_audit.csv",
        "two_paper_independent_validation_summary.csv",
    ]
    shutil.make_archive(
        "two_paper_independent_validation_results", "zip", "."
    )
    try:
        from google.colab import files
        files.download("two_paper_independent_validation_results.zip")
    except ImportError:
        print("Download is available in Google Colab.")
else:
    print("Set DOWNLOAD_RESULTS=True to download the result archive.")
